# 2. Construction heuristics with precedence
**Class:** Session 1 slides 26–44 (nearest neighbour, nearest/cheapest insertion with Δ<sub>ijk</sub> = c<sub>ik</sub> + c<sub>kj</sub> − c<sub>ij</sub>, farthest insertion), Session 8 slides 20–27 (Clarke–Wright savings s<sub>ij</sub> = c<sub>i0</sub> + c<sub>0j</sub> − c<sub>ij</sub>).

**IIMA context:** the two baselines are what students actually do: replay the timetable (B0, parallel courses) or finish one subject before the next (B1, which is cluster-first like the sweep heuristic of Session 9).

In [1]:
import os, sys
ROOT = os.path.abspath('..'); sys.path.insert(0, os.path.join(ROOT, 'src'))
from speedrun.data import load_web, Params, Profile
from speedrun.graph import build_instance
from speedrun.costs import Evaluator
web = load_web(os.path.join(ROOT, 'data', 'sample'))
P = Params.load(os.path.join(ROOT, 'data', 'sample', 'params.json'))
inst = build_instance(web, Profile(), P); ev = Evaluator(inst)
name = lambda ids: ' -> '.join(ids)
print(ev.n, 'topics,', round(ev.sum_p, 1), 'study hours (estimates)')

28 topics, 71.5 study hours (estimates)


In [2]:
from speedrun import heuristics as H
from speedrun.baselines import curriculum_order, subject_by_subject
b0, inv = curriculum_order(ev); b1 = subject_by_subject(ev)
rows = [('B0 timetable order', b0), ('B1 subject by subject', b1)] + [(k, f(ev)) for k, f in H.CONSTRUCTORS.items()]
for k, s in rows:
    bd = ev.breakdown(s)
    print(f"{k:<22} total {bd['total']:6.2f} h | switching {bd['transitions']:6.2f} | revision {bd['revision']:5.2f} | domain switches {int(bd['domain_switches'])}")

B0 timetable order     total  73.16 h | switching  -2.54 | revision  4.20 | domain switches 6
B1 subject by subject  total  70.05 h | switching  -5.39 | revision  3.94 | domain switches 3
NN                     total  72.77 h | switching  -4.44 | revision  3.71 | domain switches 4
CI                     total  74.43 h | switching  -3.29 | revision  1.22 | domain switches 6
FI                     total  74.03 h | switching  -4.79 | revision  4.32 | domain switches 4
SAV                    total  72.05 h | switching  -5.07 | revision  2.62 | domain switches 3


Worst-case bounds such as z<sup>NI</sup>/z* ≤ 2 (Session 1 slide 37) need the triangle inequality. Our c<sub>ij</sub> is asymmetric and can be negative (synergy), so we compare against exact solutions instead (notebook 1 and `scripts/scale_experiment.py`).